# Taller: Data Quality Gate para una plataforma de streaming

Este Notebook implementa el taller sobre `sample_mflix` sin modificar los documentos originales de MongoDB Atlas ni reemplazar el ETL existente. El flujo académico es:

`PROCESSED -> Quality Gate -> validos/Curated y rechazados/Quarantine`

Las validaciones principales se ejecutan con PySpark nativo. Great Expectations se comprueba de forma separada y solo se usa si es compatible con el Runtime de Databricks.

## 1. Preparacion y objetivo

Se evaluan las colecciones `movies`, `comments`, `users` y `theaters`. Se conservan los registros originales, se simulan anomalías sobre copias de trabajo y se generan métricas y Parquet de válidos y cuarentena.

In [ ]:
import os
import sys
from pathlib import Path

from pyspark.sql import functions as F

print("Python:", sys.version)
print("Spark:", spark.version)
print("Databricks:", "dbutils" in globals())

if "dbutils" in globals():
    print("Entorno Databricks detectado")
    print("Ruta del notebook:", dbutils.notebook.getContext().notebookPath().get())
else:
    print("Ejecucion local: se usara data_lake/ del repositorio")
    print("Directorio actual:", Path.cwd())

## 2. Configuracion de rutas e importacion

En Databricks Repos la ruta se detecta automáticamente desde el Notebook. Si los Parquet están en otro lugar, define `DATA_LAKE_ROOT` con una ruta real antes de ejecutar esta celda.

In [ ]:
def find_repo_root(start):
    current = Path(start).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "data_quality_gate.py").exists():
            return candidate
    raise FileNotFoundError("No se encontro src/data_quality_gate.py en Databricks Repos")

if "dbutils" in globals():
    notebook_path = dbutils.notebook.getContext().notebookPath().get()
    if notebook_path.startswith("/Workspace/"):
        repo_root = find_repo_root(notebook_path)
    else:
        repo_root = find_repo_root(Path("/Workspace") / notebook_path.lstrip("/"))
else:
    repo_root = find_repo_root(Path.cwd())

sys.path.insert(0, str(repo_root))
from src.data_quality_gate import (
    metrics_dataframe,
    simulate_dirty_data,
    validate_all_collections,
    write_quality_outputs,
)

lake_root = os.environ.get("DATA_LAKE_ROOT", str(repo_root / "data_lake"))
print("Repositorio:", repo_root)
print("Data Lake:", lake_root)

def read_parquet_path(path):
    if "dbutils" in globals():
        return spark.read.parquet(path)
    local_files = [str(file) for file in Path(path).rglob("*.parquet")]
    if not local_files:
        raise FileNotFoundError(f"No se encontraron archivos Parquet en {path}")
    return spark.read.parquet(*local_files)

if "dbutils" in globals():
    print("Rutas PROCESSED esperadas:")
    for collection in ("movies", "comments", "users", "theaters"):
        print(f"{lake_root}/processed/{collection}")

## 3. Colecciones, esquema y exploracion

La fuente de esta ejecucion es la zona PROCESSED ya existente. Esto evita duplicar la ingesta y conserva el flujo funcional del proyecto.

In [ ]:
collection_names = ["movies", "comments", "users", "theaters"]
frames = {}
for collection in collection_names:
    path = f"{lake_root}/processed/{collection}"
    frame = read_parquet_path(path)
    frames[collection] = frame
    print(f"{collection}: {frame.count()} registros")
    frame.printSchema()
    display(frame.limit(5))

## 4. Simulacion controlada de datos sucios

La funcion trabaja sobre copias de los DataFrames. No ejecuta operaciones de escritura, actualizacion o eliminacion en MongoDB Atlas.

In [ ]:
dirty_frames = {
    collection: simulate_dirty_data(frame, collection)
    for collection, frame in frames.items()
}
for collection, frame in dirty_frames.items():
    print(f"Anomalias simuladas: {collection}")
    display(frame.where(F.col("_simulated_dirty_record") == True))

## 5. Quality Gate con PySpark nativo

Reglas utilizadas: completitud, validez, unicidad, consistencia e integridad referencial. Los errores quedan identificados por nombre en `quality_errors`.

In [ ]:
results = validate_all_collections(dirty_frames)
for collection, result in results.items():
    print(f"=== {collection.upper()} ===")
    print(result.metrics)
    print("Registros validos")
    display(result.valid_df)
    print("Registros rechazados")
    display(result.rejected_df)
    print("Incumplimientos por regla")
    display(result.rule_counts_df)

## 6. Separacion de Curated y Quarantine

Los DataFrames validos son la entrada aprobada para Curated. Los rechazos se conservan con identificador, registro original, procedencia, reglas y fecha de validacion.

In [ ]:
quality_root = f"{lake_root}/quality_gate"
if "dbutils" in globals():
    paths = write_quality_outputs(results, quality_root, mode="overwrite")
else:
    # Spark write en Windows requiere winutils; se usa PyArrow solo para esta prueba local.
    import json
    import pyarrow as pa
    import pyarrow.parquet as pq
    import shutil
    local_root = Path(quality_root)
    shutil.rmtree(local_root, ignore_errors=True)
    def write_local_parquet(frame, output_path):
        output_path.mkdir(parents=True, exist_ok=True)
        rows = [json.loads(value) for value in frame.toJSON().collect()]
        if rows:
            table = pa.Table.from_pylist(rows)
        else:
            table = pa.table({name: pa.array([], type=pa.string()) for name in frame.columns})
        pq.write_table(table, output_path / "part-00000.parquet")

    paths = {}
    for collection, result in results.items():
        local_outputs = {
            f"valid_{collection}": (result.valid_df, local_root / "valid" / collection),
            f"quarantine_{collection}": (result.quarantine_df, local_root / "quarantine" / collection),
            f"rules_{collection}": (result.rule_counts_df, local_root / "metrics" / f"{collection}_rules"),
        }
        for name, (frame, output_path) in local_outputs.items():
            write_local_parquet(frame, output_path)
            paths[name] = str(output_path)
    summary_path = local_root / "metrics" / "run_summary"
    write_local_parquet(metrics_dataframe(results), summary_path)
    paths["run_summary"] = str(summary_path)
print("Salidas generadas:")
for name, path in paths.items():
    print(name, "->", path)

print("Metricas consolidadas")
display(read_parquet_path(paths["run_summary"]))
print("Ejemplo de cuarentena")
display(read_parquet_path(paths["quarantine_movies"]).limit(20))

## 7. Verificacion de lectura Parquet

Cada salida se vuelve a leer con Spark para comprobar que la persistencia es válida.

In [ ]:
parquet_checks = []
for name, path in paths.items():
    check = read_parquet_path(path)
    parquet_checks.append((name, check.count()))
print("Lecturas verificadas:", parquet_checks)

## 8. Great Expectations: comprobacion de compatibilidad

Las validaciones anteriores son PySpark nativo y no deben presentarse como GX. Esta celda solo determina si GX está disponible en el Runtime actual.

In [ ]:
import importlib.util
gx_spec = importlib.util.find_spec("great_expectations")
gx_available = gx_spec is not None
print("Great Expectations disponible:", gx_available)
if gx_available:
    import great_expectations as gx
    print("Version GX:", gx.__version__)
    print("Accion: crear la Expectation Suite con la API de esta version y guardar Data Docs en un Volume.")
else:
    print("GX no esta instalado. No se instala automaticamente ni se declara como implementado.")

## 9. Entrada aprobada para Curated

Esta ejecución académica permite que solo los registros válidos continúen al modelo analítico. El ETL productivo de `main.py` no se modifica.

In [ ]:
valid_frames = {
    collection: result.valid_df
    for collection, result in results.items()
}
for collection, frame in valid_frames.items():
    print(collection, frame.count(), "registros aprobados para Curated")

## 10. Reflexion academica

Permitir películas sin título, con calificaciones fuera del rango esperado o con duraciones inválidas contaminaría los indicadores de calidad del catálogo. Además, un comentario asociado a una película inexistente rompería la integridad de las relaciones utilizadas en la analítica. Estos errores podrían distorsionar los análisis por género, las métricas de interacción y cualquier recomendación basada en el catálogo. El Quality Gate evita que esos registros lleguen a Curated sin trazabilidad y conserva los rechazos para su revisión.

## Evidencias para entregar

Capturar: versión de Python y Spark; Volumes; colecciones y esquemas; datos sucios simulados; reglas y errores; DataFrames válidos y rechazados; cuarentena; métricas consolidadas; escritura y lectura Parquet; y Data Docs HTML si GX es compatible y se ejecutó correctamente.